# Participant-Safe Parkinson's Voice Classification

**Recruiter-facing end-to-end analysis · Grouped high-stakes classification · Python 3.12/3.13**

> With no participant overlap, the eight-person holdout has balanced accuracy 0.750, sensitivity 1.000, and specificity 0.500; uncertainty is necessarily wide.

## Executive summary

**Objective:** Evaluate voice-based classification without allowing recordings from one participant to cross train/test boundaries.

**Data:** 195 recordings, 22 acoustic features, and 32 derived participant identifiers.

**Verified result:** With no participant overlap, the eight-person holdout has balanced accuracy 0.750, sensitivity 1.000, and specificity 0.500; uncertainty is necessarily wide.

**Decision supported:** Judge whether signal warrants external study—not make a clinical decision.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** A biomedical ML researcher reviewing methodology.

**Decision:** Judge whether signal warrants external study—not make a clinical decision.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '11-parkinsons-disease-detection'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.10
pandas            2.3.3
NumPy             2.5.2
SciPy            1.18.0
scikit-learn      1.9.0
Matplotlib       3.11.1

Project: 11-parkinsons-disease-detection


## 4. Data provenance and scope

Matches the UCI Parkinsons voice dataset; participant IDs are derived from the recording-name field.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

          file  size_mb           sha256
parkinsons.csv    0.038 0736fa3a8ac098c4


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


parkinsons.csv: 24 columns
          name  MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  MDVP:Jitter(Abs)  ...     RPDE      DFA   spread1  spread2       D2      PPE
phon_R01_S01_1      119.992       157.302        74.997         0.00784           0.00007  ... 0.414783 0.815285 -4.813031 0.266482 2.301442 0.284654
phon_R01_S01_2      122.400       148.650       113.819         0.00968           0.00008  ... 0.458359 0.819521 -4.075192 0.335590 2.486855 0.368674
phon_R01_S01_3      116.682       131.111       111.555         0.01050           0.00009  ... 0.429895 0.825288 -4.443179 0.311173 2.342259 0.332634
phon_R01_S01_4      116.676       137.871       111.366         0.00997           0.00009  ... 0.434969 0.819235 -4.117501 0.334147 2.405554 0.368975
phon_R01_S01_5      116.014       141.781       110.655         0.01284           0.00011  ... 0.417356 0.823484 -3.747787 0.234513 2.332180 0.410335


## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 135 lines
Functions: _participant_table, _bootstrap_interval, run_analysis


## 7. Methodology and hypotheses

Participant-level holdout, StratifiedGroupKFold selection, dummy/logistic/SVM/forest/boosting comparison, recording and participant metrics, bootstrap intervals, false-negative review, and permutation importance.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_11_parkinsons_disease_detection", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

D:\AI-Training\applied-data-science\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\AI-Training\applied-data-science\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\AI-Training\applied-data-science\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\AI-Training\applied-data-science\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` param

## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


data_quality.csv (24 fields)
          column   dtype  missing_count  missing_percent  unique_values  constant
            name  object              0              0.0            195     False
     MDVP:Fo(Hz) float64              0              0.0            195     False
    MDVP:Fhi(Hz) float64              0              0.0            195     False
    MDVP:Flo(Hz) float64              0              0.0            195     False
  MDVP:Jitter(%) float64              0              0.0            173     False
MDVP:Jitter(Abs) float64              0              0.0             19     False
        MDVP:RAP float64              0              0.0            155     False
        MDVP:PPQ float64              0              0.0            165     False
      Jitter:DDP float64              0              0.0            180     False
    MDVP:Shimmer float64              0              0.0            188     False
MDVP:Shimmer(dB) float64              0              0.0            

## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'participant_grouped_cv.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'With no participant overlap, the eight-person holdout has balanced accuracy 0.750, sensitivity 1.000, and specificity 0.500; uncertainty is necessarily wide.')

Primary evidence: participant_grouped_cv.csv, shape=(5, 5)
                 model  participant_cv_roc_auc_mean  participant_cv_roc_auc_std  participant_cv_balanced_accuracy_mean  participant_cv_recall_mean
     gradient_boosting                       0.7812                      0.1375                                 0.6000                      0.9500
   logistic_regression                       0.5875                      0.4211                                 0.5750                      0.7750
         random_forest                       0.5500                      0.3894                                 0.5938                      0.9375
support_vector_machine                       0.5375                      0.4888                                 0.5938                      0.9375
      dummy_prevalence                       0.5000                      0.1021                                 0.5000                      1.0000

Verified result:
With no participant overlap, the eight-pe

## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])


VALIDATION
{
  "train_participants": 24,
  "held_out_participants": 8,
  "same_participant_in_train_and_test": false,
  "model_selection": "4-fold StratifiedGroupKFold on training participants",
  "random_seed": 42
}

PARTICIPANT_BOOTSTRAP_INTERVALS
{
  "participant_roc_auc_95_percent_bootstrap": [
    1.0,
    1.0
  ],
  "participant_balanced_accuracy_95_percent_bootstrap": [
    0.5,
    1.0
  ],
  "participant_recall_95_percent_bootstrap": [
    1.0,
    1.0
  ]
}


## 12. Visual evidence

### Participant Confusion Matrix

![participant_confusion_matrix](../reports/figures/participant_confusion_matrix.png)

### Participant Safe Model Evidence

![participant_safe_model_evidence](../reports/figures/participant_safe_model_evidence.png)

## 13. Business interpretation

With no participant overlap, the eight-person holdout has balanced accuracy 0.750, sensitivity 1.000, and specificity 0.500; uncertainty is necessarily wide.

The correct action is to use this result as evidence for **Judge whether signal warrants external study—not make a clinical decision.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

Educational evaluation only—not a diagnostic, screening, monitoring, or treatment system. The eight-person holdout and absent external validation preclude clinical use.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                           artifact  size_kb       sha256
   reports\figures\participant_confusion_matrix.png     26.2 5fdf8ad757fe
reports\figures\participant_safe_model_evidence.png    139.0 e906014f30e8
                               reports\metrics.json      4.4 782a416e321d
                    reports\tables\data_quality.csv      0.9 8dbce4ed4ffd
reports\tables\held_out_participant_predictions.csv      0.3 4c8d089de542
          reports\tables\participant_grouped_cv.csv      0.4 4310516a46ee
          reports\tables\permutation_importance.csv      1.0 2e61ed4be2ae


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed evaluate voice-based classification without allowing recordings from one participant to cross train/test boundaries. using participant-level holdout, stratifiedgroupkfold selection, dummy/logistic/svm/forest/boosting comparison, recording and participant metrics, bootstrap intervals, false-negative review, and permutation importance. The final verified conclusion is: **With no participant overlap, the eight-person holdout has balanced accuracy 0.750, sensitivity 1.000, and specificity 0.500; uncertainty is necessarily wide.** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/11-parkinsons-disease-detection/src/analysis.py
python scripts/execute_notebooks.py --project 11-parkinsons-disease-detection
```